In [1]:
!pip install -U pip setuptools wheel
!pip install -U spacy
!python -m spacy download en_core_web_sm
!pip install git+https://github.com/medianeuroscience/emfdscore.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 48.2 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
  Cloning https://github.com/medianeuroscience/emfdscore.git to /tmp/pip-req-build-e1e8gx1l
  Running command git clone --filter=blob:none --quiet https://github.com/medianeuroscience/emfdscore.git /tmp/pip-req-build-e1e8gx1l
  Resolved https://github.com/medianeuroscience/emfdscore.git to commit 5523d76c25afa65a59ca00e44efe45885499b116
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import re
import pandas as pd
from emfdscore.scoring import score_docs

toxicity_samples = pd.read_csv("/content/drive/MyDrive/CS74-civilcomments/toxicity_samples.csv")

IDENTITY_TERMS = ["gay", "gays", "lesbian", "lesbians", "straight", "transgender", "trans",
                  "christian", "christians", "muslim", "muslims", "jewish", "jews",
                  "black", "blacks", "white", "whites", "asian", "asians", "latino", "latinos",
                  "women", "woman", "men", "man", "female", "male"]
pattern = re.compile(r"\b(" + "|".join(IDENTITY_TERMS) + r")\b", flags=re.IGNORECASE)

def mask_identity(text):
    # replace with a word not in the moral foundations dictionary
    return pattern.sub("xxidenxx", str(text))

toxicity_samples["comment_masked"] = toxicity_samples["comment_text"].apply(mask_identity)

NLTK stopwords missing, downloading now.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
# eMFD expects df with text in column 0 for scoring:
docs = pd.DataFrame(toxicity_samples["comment_masked"].tolist())

emfd = score_docs(docs,
                  'emfd',         # dictionary type
                  'all',          # use all five foundation probabilities per word
                  'bow',          # bag-of-words scoring
                  'vice-virtue',  # output metric (see below)
                  len(docs))

emfd.columns = ["emfd_" + c for c in emfd.columns]
emfd.index = toxicity_samples.index
toxicity_samples_scored = pd.concat([toxicity_samples, emfd], axis=1)
toxicity_samples_scored.to_csv("/content/drive/MyDrive/CS74-civilcomments/toxicity_samples_emfd.csv", index=False)

Processed: 20000 100% |❤❤❤❤❤❤❤❤❤❤❤❤❤❤❤❤❤❤| Elapsed Time: 0:02:35 Time:  0:02:35
